### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_ZYEXYLLCPSNH


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250521_squid_cal520"
recname = "rec4"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec4"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_yutaka_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_yutaka/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719500,) (10948,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(190, 719500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.524506,0.960489,0.670964,0.641650,0.935599,1.126987,1.273344,1.368839,...,-0.557304,-0.805001,-1.014531,-0.796999,-0.756649,-1.059094,-1.203153,-0.547385,-0.702084,-0.775541
1,0,1,0.493075,-0.253646,-0.421551,-0.287608,-0.219770,-0.268780,-0.335196,-0.191671,...,0.875241,0.965176,0.815074,0.823305,0.554849,0.492000,1.056166,0.881324,0.666366,1.122832
2,0,2,0.589718,-1.001396,-1.143333,-0.826768,-0.744494,-0.758819,-1.092078,-0.903668,...,-1.011123,-0.936627,-1.171870,-1.200557,-1.055442,-0.900780,-0.752569,-0.675521,-0.770959,-0.898372
3,0,3,0.233257,0.121231,0.237911,0.346188,0.439113,0.223548,0.175290,0.651539,...,-0.241559,-0.193570,-0.585435,-0.871878,-0.599670,-0.111751,-0.221592,-0.339634,-0.788378,-0.770752
4,0,4,0.607708,1.216921,1.110502,0.863997,1.096252,1.212157,1.286351,1.446748,...,-1.282236,-1.057369,-1.075751,-1.146461,-1.148168,-1.187980,-0.869246,-0.977085,-0.965698,-0.886925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,0,185,0.139798,0.257176,0.222617,0.124275,0.360076,0.351546,-0.034712,-0.051853,...,-0.179025,-1.017373,-0.822905,-0.615978,-0.688417,-0.640660,-0.551754,-0.480556,0.024780,-0.017314
186,0,186,0.322012,-0.634072,-0.883090,-0.829899,-0.610751,-0.486314,-0.606130,-0.523600,...,-0.390439,-0.545258,-0.683469,-0.696230,-0.365992,-0.650135,-0.740752,-0.648869,-0.523796,-0.813068
187,0,187,0.586264,0.453577,0.574693,0.483344,0.311286,0.371113,0.416181,0.247211,...,-1.351367,-1.355669,-1.613227,-1.437661,-1.154828,-1.321202,-1.481805,-1.629170,-1.482048,-1.318691
188,0,188,0.104200,0.379859,0.590028,0.383120,1.063987,0.483567,0.050842,-0.114120,...,0.051755,0.263304,0.080684,-0.150813,-0.423769,-0.265310,-0.474156,-0.175510,0.172831,0.032232


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec4"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_ZYEXYLLCPSNH/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec4"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6
Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_ZYEXYLLCPSNH/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250521_rec4"', returncode=0)